<div align="center">

# Day 01: Concept and Architecture

**Digital Twin with Python (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/digital-twin-veltech-NB-lecture/tree/main/days/day-01) · [Lecture page](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html) · [Interactive lab](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lab.html) · [PDF notes](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/Day01_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows; they have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from a Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures, in a few seconds to a few minutes. When the notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field, asdict
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.6), 'axes.grid': True, 'grid.alpha': 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback. Without an expected value in the
    call, check takes the value and its tolerance from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok



# The asset: a 48 V lithium-ion battery pack under a duty cycle.
#
#   states   SoC    state of charge, 0 to 1
#            T      cell temperature in degrees Celsius
#            Q      usable capacity in Ah, fades with use
#            R      internal resistance in ohm, grows as capacity fades
#   inputs   I(t)   current demanded by the load in A, positive when discharging
#            u      cooling-fan duty, 0 to 1
#            T_amb  ambient temperature in degrees Celsius
#
# The state of charge changes over minutes, the temperature over tens of minutes and
# the capacity over months.

@dataclass
class PackParams:
    Q0: float = 60.0        # Ah, capacity when new
    R0: float = 0.045       # Ohm, internal resistance when new
    m_cp: float = 9000.0    # J/K, thermal mass of the pack
    hA0: float = 2.2        # W/K, passive convection
    hA1: float = 12.0       # W/K, extra convection at full fan
    k_age: float = 5.5e-8   # Ah lost per A-second at the reference temperature
    Ea_R: float = 3800.0    # K, Arrhenius slope of the ageing law
    T_ref: float = 298.15   # K
    fan_W: float = 25.0     # W, electrical cost of the fan at full duty

    def as_vector(self):
        return np.array([self.R0, self.m_cp, self.hA0, self.hA1])


def pack_derivatives(t, x, I, u, T_amb, p: PackParams):
    """dx/dt for x = [SoC, T, Q, R]. Pure physics, no data, no noise."""
    SoC, T, Q, R = x
    T_K = T + 273.15
    P_gen = I ** 2 * R                                    # ohmic heating, W
    P_cool = (p.hA0 + p.hA1 * u) * (T - T_amb)            # convective loss, W
    arrh = np.exp(-p.Ea_R * (1.0 / T_K - 1.0 / p.T_ref))  # ageing accelerates with heat
    dSoC = -I / (3600.0 * max(Q, 1e-6))
    dT = (P_gen - P_cool) / p.m_cp
    dQ = -p.k_age * abs(I) * arrh
    dR = p.R0 * 0.9 * (p.Q0 - Q) / p.Q0 * 1e-3            # resistance tracks capacity fade
    return np.array([dSoC, dT, dQ, dR])


def duty_cycle(t, kind="urban", seed=RANDOM_STATE):
    """Current demand in amperes as a function of time in seconds."""
    rng = np.random.default_rng(seed)
    if kind == "urban":       # stop-start: bursts of traction, regenerative braking
        base = 14 + 26 * (np.sin(2 * np.pi * t / 420.0) > 0.15)
        regen = -18 * (np.sin(2 * np.pi * t / 420.0 + 0.9) > 0.93)
        return base + regen
    if kind == "highway":     # steady high load
        return 40 + 6 * np.sin(2 * np.pi * t / 1800.0)
    if kind == "idle":
        return 8 + 0 * t
    raise ValueError(kind)



def rk4_step(fun, t, x, dt, *args):
    k1 = fun(t, x, *args)
    k2 = fun(t + dt / 2, x + dt / 2 * k1, *args)
    k3 = fun(t + dt / 2, x + dt / 2 * k2, *args)
    k4 = fun(t + dt, x + dt * k3, *args)
    return x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def simulate(p: PackParams, T_end=3600.0, dt=1.0, x0=None, cycle="urban",
             controller=None, T_amb=25.0, seed=RANDOM_STATE):
    """Roll the physical asset forward. `controller(t, x) -> fan duty` closes the loop."""
    x = np.array([0.95, T_amb + 1.0, p.Q0, p.R0]) if x0 is None else np.array(x0, float)
    n = int(T_end / dt)
    ts = np.arange(n + 1) * dt
    X = np.zeros((n + 1, 4)); X[0] = x
    U = np.zeros(n + 1); I_log = np.zeros(n + 1)
    for k in range(n):
        t = ts[k]
        I = float(duty_cycle(t, cycle, seed))
        u = 0.0 if controller is None else float(np.clip(controller(t, x), 0, 1))
        U[k] = u; I_log[k] = I
        x = rk4_step(pack_derivatives, t, x, dt, I, u, T_amb, p)
        x[0] = np.clip(x[0], 0.0, 1.0)
        X[k + 1] = x
    U[-1] = U[-2]; I_log[-1] = I_log[-2]
    return pd.DataFrame({"t": ts, "SoC": X[:, 0], "T": X[:, 1], "Q": X[:, 2],
                         "R": X[:, 3], "u": U, "I": I_log})



@dataclass
class SensorSpec:
    """Turns a true value into a reading: noise, bias, drift, quantisation, dropout and latency."""
    noise_sd: float = 0.0       # gaussian noise, in sensor units
    bias: float = 0.0           # constant offset
    drift_per_hour: float = 0.0 # slow calibration drift
    quantum: float = 0.0        # ADC quantisation step, 0 = none
    dropout: float = 0.0        # probability a sample is lost
    latency_s: float = 0.0      # transport delay


def read_sensor(t, true_values, spec: SensorSpec, rng):
    v = np.asarray(true_values, float).copy()
    v = v + spec.bias + spec.drift_per_hour * (t / 3600.0)
    if spec.noise_sd:
        v = v + rng.normal(0, spec.noise_sd, v.shape)
    if spec.quantum:
        v = np.round(v / spec.quantum) * spec.quantum
    if spec.dropout:
        v = np.where(rng.random(v.shape) < spec.dropout, np.nan, v)
    return v


print('pack model, integrator and sensors ready')

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 1.1": (lambda: 960.0, 1e-6),
    "Exercise 1.2": (lambda: 3, 1e-6),
    "Exercise 1.3": (lambda: 41.17, 1e-6),
    "Exercise A": (lambda: int(sweep.loc[sweep["shadow_RMSE_C"].idxmin(), "sync_interval_s"]), 1e-6),
    "Exercise B": (lambda: float(grade.set_index("scenario").loc["sensor bias -5 C", "truly_hot_s"]), 1e-6),
}

## Python step 1: Values, variables, lists and decisions

A notebook is a sequence of cells. Text cells explain, and code cells hold Python instructions that run when the cell is executed with Shift+Enter, with the output shown directly below the cell &#91;[10](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#ref-10)&#93;. This step works with the temperature of the battery pack: It computes five steps of the thermal model by hand and ends with the pack as an object of a class &#91;[6](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#ref-6)&#93;.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-01/NB01_concept_and_architecture.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

### Values and variables

In [ ]:
# Python practice: a small example for learning the language
soc = 0.82              # state of charge, 0 to 1
temperature_C = 41.3    # cell temperature in degrees Celsius
current_A = 40.0        # current drawn by the load
chemistry = "lithium-ion"
print(type(soc), type(chemistry))
print("SoC:", soc, "temperature:", temperature_C)

A variable is a name bound to a value with `=`. Numbers with a decimal point are floats, text in quotation marks is a string, and `True` and `False` are booleans. Units belong in the names or in a comment after `#`, because a number without a unit is a source of errors in every engineering program.

### A decision is an if statement

In [ ]:
# Python practice: a small example for learning the language
fan_on_C = 42.0
if temperature_C > fan_on_C:
    fan = "on"
else:
    fan = "off"
print(f"Temperature {temperature_C:.1f} °C, threshold {fan_on_C:.1f} °C, fan {fan}")

An `if` statement runs the indented block below it only when its condition is true, and `else` covers the remaining case. An f-string, written with `f` before the quotation mark, inserts the value of each expression in braces, and `:.1f` prints one decimal. This rule is the simplest controller of the pack.

### Lists and loops

In [ ]:
# Python practice: a small example for learning the language
readings = [36.0, 39.4, 42.1, 44.8, 46.9, 48.2]
hot = 0
for minute, t in zip(range(0, 60, 10), readings):
    status = "over limit" if t > 46 else "ok"
    hot = hot + (t > 46)
    print(f"minute {minute:2d}: {t:4.1f} °C  {status}")
print("readings over 46 °C:", hot)

A list holds values in order between square brackets. `range(0, 60, 10)` produces 0, 10, 20 and so on, and `zip` walks two sequences in step: `for minute, t in ...` gives the two values of each step the names `minute` and `t`. The one-line form `a if condition else b` chooses between two values, and adding a boolean to a number counts it as 1 or 0. In the f-string, `:2d` and `:4.1f` reserve two and four places, so that the columns line up.

### A simulation is a loop

In [ ]:
# Python practice: a small example for learning the language
T, T_amb = 36.0, 35.0                 # pack and ambient temperature
R, m_cp, hA = 0.045, 9000.0, 2.2      # ohm, J/K, W/K
dt = 60.0                             # one minute in seconds
for minute in range(1, 6):
    heat_in = current_A ** 2 * R
    heat_out = hA * (T - T_amb)
    T = T + dt * (heat_in - heat_out) / m_cp
    print(f"after {minute} min: {T:.2f} °C")

Each pass of the loop applies the energy balance of the lecture for one minute: The new temperature is the old one plus the time step times the rate of change. This is the explicit Euler method, and Day 2 measures exactly how accurate it is. The operator `**` raises to a power.

### Functions and dictionaries

In [ ]:
# Python practice: a small example for learning the language
def fan_duty(T, on_C=42.0, span_C=6.0):
    """Proportional fan command between 0 and 1."""
    return min(1.0, max(0.0, (T - on_C) / span_C))

reading = {"T": 47.3, "SoC": 0.64, "I": 41.0}
reading["fan"] = fan_duty(reading["T"])
for key, value in reading.items():
    print(f"{key:4s} = {value:.2f}")

`def` defines a function with parameters, default values and a `return` statement, and the text in triple quotation marks documents it. `min` and `max` clip the command to the range from 0 to 1. A dictionary maps keys to values: `reading["T"]` looks a value up, assigning to a new key adds an entry, and `items` returns key and value pairs for a loop. A sensor message is naturally a dictionary.

### A pack as an object

In [ ]:
# Python practice: a small example for learning the language
class Pack:
    """A pack that keeps its temperature and advances it one time step at a time."""
    def __init__(self, T, current_A):
        self.T = T                      # temperature in degrees Celsius
        self.current_A = current_A      # current drawn by the load
    def step(self, dt):
        heat_in = self.current_A ** 2 * R
        heat_out = hA * (self.T - T_amb)
        self.T = self.T + dt * (heat_in - heat_out) / m_cp

pack = Pack(36.0, current_A)
for minute in range(1, 6):
    pack.step(dt)
print(f"after 5 min: {pack.T:.2f} °C")

A class bundles values with the functions that use them. `Pack(36.0, current_A)` creates one object of the class: Python runs `__init__`, which stores the two values on `self`, the object itself. Values stored on `self` are called attributes, and functions defined inside the class are called methods. Every method receives the object as its first argument `self`, so `step` reads and changes the temperature of its own pack, and `pack.step(dt)` calls it. `pack.T` reads an attribute from outside. The object repeats the five steps of the loop above, with `R`, `hA`, `T_amb` and `m_cp` from that cell, and ends at the same temperature. `@dataclass` in front of a class, as in front of `PackParams` in the setup cell, writes such an `__init__` automatically from a list of attributes with default values. The notebook later builds the five parts of the twin as classes of this kind.

**Quick check.** What does `fan_duty(39.0)` return, and why?

<details><summary>Answer</summary>

`0.0`. The expression `(39 - 42) / 6` is negative, and `max(0.0, ...)` clips it to zero, so the fan stays off.

</details>

### Exercises for Python step 1

**Exercise 1.1.** A 48 V pack delivers 40 A for 30 minutes. Store the delivered energy in watt-hours, voltage times current times hours, in `energy_Wh`.

In [ ]:
energy_Wh = None   # your answer

check("Exercise 1.1", energy_Wh)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
energy_Wh = 48 * 40 * 0.5

check("Exercise 1.1", energy_Wh)

**Exercise 1.2.** Write a function `over_limit(values, limit)` that returns how many values are above `limit`, and store `over_limit(readings, 44)` in `answer_12`.

In [ ]:
def over_limit(values, limit):
    return None   # your code

answer_12 = over_limit(readings, 44)

check("Exercise 1.2", answer_12)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def over_limit(values, limit):
    return sum(1 for v in values if v > limit)

answer_12 = over_limit(readings, 44)

check("Exercise 1.2", answer_12)

**Exercise 1.3.** Repeat the five-minute simulation with a current of 60 A, starting again from 36 °C. Store the final temperature, rounded to two decimals with `round(T, 2)`, in `T5`.

In [ ]:
T5 = None   # your code

check("Exercise 1.3", T5)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
T = 36.0
for minute in range(1, 6):
    T = T + dt * (60.0 ** 2 * R - hA * (T - T_amb)) / m_cp
T5 = round(T, 2)

check("Exercise 1.3", T5)

## 1. The pack and its datasheet twin

*Lecture: [The asset of the course](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#the-asset-of-the-course)*

The pack is simulated with the model of the lecture. **Python move: dataclasses as parameter sets.** `PackParams()` holds the parameters of the pack, and `PackParams(R0=0.038)` creates a variant with one value changed.

**From the formulas to the code.** `TRUE` and `BELIEF` are two sets of the parameters $\theta$: the pack as built and its datasheet version, which the virtual side uses from section 2 on. `simulate` advances the state $x = (\mathrm{SoC}, T, Q, R)$, the array `x`, in 7200 steps of `dt = 1.0` second with `rk4_step`, the classical Runge-Kutta method of order four (RK4) of the lecture. `pack_derivatives` is the rule $f$, and its lines `dSoC`, `dT`, `dQ` and `dR` are the four equations of the lecture. In them, `P_gen` is the ohmic heating $I^2 R$, `P_cool` the convection and `arrh` the Arrhenius factor. The current is read once per step from `duty_cycle` and held during the step, and `max(Q, 1e-6)` guards the division by the capacity. After every step `np.clip` keeps the state of charge (SoC) between 0 and 1, which holds it at zero from minute 85 on. The table prints every 1200th row, one line per 20 minutes, and the capacity lost is `TRUE.Q0 - df_true.Q.iloc[-1]`, the difference $Q_0 - Q$ at the end of the trip.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "answer_12" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-01/NB01_concept_and_architecture.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
# The twin starts from datasheet parameters, which differ from those of the physical pack.
TRUE = PackParams()                                   # the physical pack, as built
BELIEF = PackParams(R0=0.038, m_cp=10200.0, hA0=2.9)  # what the datasheet claims

CYCLE, T_AMB = "highway", 35.0        # a hot afternoon run at 35 degrees ambient
df_true = simulate(TRUE, T_end=7200, dt=1.0, cycle=CYCLE, T_amb=T_AMB)
print(f"physical pack over a 2-hour {CYCLE} trip at {T_AMB:.0f} C ambient")
print(df_true.iloc[::1200][["t", "SoC", "T", "Q", "I"]].round(3).to_string(index=False))
print(f"\nmean current {df_true.I.mean():.1f} A   peak temperature {df_true['T'].max():.1f} C"
      f"   capacity lost {TRUE.Q0 - df_true.Q.iloc[-1]:.4f} Ah")

**Python note.** `iloc[::1200]` is a slice with a step: It starts at the first row, ends at the last and takes every 1200th row. A list of names in double brackets, `[["t", "SoC", "T", "Q", "I"]]`, selects several columns at once. `round(3)` rounds every number to three decimals, and `to_string(index=False)` prints the table without its row numbers.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3.1))
ax[0].plot(df_true["t"] / 60, df_true["I"], color="#7f8c8d"); ax[0].set_title("current drawn (A)", fontsize=9)
ax[1].plot(df_true["t"] / 60, df_true["T"], color="#c0392b"); ax[1].axhline(46, ls="--", color="k", lw=.8)
ax[1].set_title("cell temperature (C), limit 46", fontsize=9)
ax[2].plot(df_true["t"] / 60, df_true["SoC"], color="#2980b9"); ax[2].set_title("state of charge", fontsize=9)
for a in ax: a.set_xlabel("time (min)")
plt.tight_layout(); plt.show()

**Python note.** `plt.subplots(1, 3, figsize=(12, 3.1))` creates a figure with three panels side by side, `ax[0]` to `ax[2]`, and `figsize` gives its width and height in inches as a tuple, a fixed group of values in round brackets. `df_true["t"] / 60` divides the whole column at once and turns seconds into minutes, and `for a in ax:` labels the time axis of each panel.

**Observe and reflect.** The state of charge reaches zero after about 85 minutes, yet the temperature keeps rising until the end. What does the model assume about the current once the pack is empty, and what would a real battery management system do at that moment? Which panels would change if the current stopped at a state of charge of 0.05?

## 2. The maturity ladder &#91;[2](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#ref-2)&#93;

*Lecture: [What each arrow buys](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#what-each-arrow-buys)*

**From the formulas to the code.** `run_ladder` advances five state vectors with `rk4_step`. `x_plant_a` is the pack without control and `x_plant_b` the pack whose fan the twin sets, both with the parameters `TRUE`. `x_model`, `x_shadow` and `x_twin` are the three beliefs $\hat{x}$, with the parameters `BELIEF`. Plant is the word of control engineering for the asset, and the comments `P->D` and `D->P` mark the two arrows, physical to digital and digital to physical. The condition `(k + 1) % int(sync_every / dt) == 0` is true every 120 steps: `read_sensor` then returns the reading $y_k$ with bias, noise and rounding, and `x_shadow[1] = T_meas` is the overwriting $\hat{T} \leftarrow y_k$. A lost reading arrives as `nan`, and `np.isfinite` then skips the update. `cooling_policy` is the fan rule $u = \mathrm{clip}((\hat{T} - 42) / 6, 0, 1)$, and `rmse` is the root mean square error (RMSE). All five vectors start from the same values, including the resistance `TRUE.R0`. In sections 2 to 4 the beliefs therefore differ from the pack through `m_cp` and `hA0`, and the datasheet resistance only enters the slow growth of $R$.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "fig" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-01/NB01_concept_and_architecture.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
TEMP_SENSOR = SensorSpec(noise_sd=0.35, bias=0.2, quantum=0.25, dropout=0.01)

def run_ladder(T_end=7200.0, dt=1.0, sync_every=120.0, cycle=None, T_amb=None):
    """One plant, three virtual counterparts.

    digital model  : open-loop simulation from initial conditions. No arrows at all.
    digital shadow : plant -> twin state sync every `sync_every` seconds. One arrow.
    digital twin   : the shadow, plus the twin's controller driving the plant's fan. Two arrows.
    """
    cycle = cycle or CYCLE; T_amb = T_AMB if T_amb is None else T_amb
    n = int(T_end / dt)
    ts = np.arange(n + 1) * dt

    x_plant_a = np.array([0.95, T_amb + 1.0, TRUE.Q0, TRUE.R0])   # plant for the shadow experiment
    x_plant_b = x_plant_a.copy()                           # plant for the twin experiment
    x_model = x_plant_a.copy()                             # open-loop virtual counterpart
    x_shadow = x_plant_a.copy()
    x_twin = x_plant_a.copy()

    def cooling_policy(T_hat):
        """The twin's decision rule. It sees only its own estimate of the temperature."""
        return float(np.clip((T_hat - 42.0) / 6.0, 0.0, 1.0))

    log = {k: np.zeros((n + 1, 4)) for k in ("plant_a", "plant_b", "model", "shadow", "twin")}
    for k, v in log.items():
        v[0] = {"plant_a": x_plant_a, "plant_b": x_plant_b, "model": x_model,
                "shadow": x_shadow, "twin": x_twin}[k]
    u_log = np.zeros(n + 1)

    for k in range(n):
        t = ts[k]
        I = float(duty_cycle(t, cycle))

        # ---- experiment A: the plant runs in open loop while the model and the shadow observe
        x_plant_a = rk4_step(pack_derivatives, t, x_plant_a, dt, I, 0.0, T_amb, TRUE)
        x_model = rk4_step(pack_derivatives, t, x_model, dt, I, 0.0, T_amb, BELIEF)
        x_shadow = rk4_step(pack_derivatives, t, x_shadow, dt, I, 0.0, T_amb, BELIEF)
        if (k + 1) % int(sync_every / dt) == 0:                 # <-- the automatic P->D arrow
            T_meas = read_sensor(t, [x_plant_a[1]], TEMP_SENSOR, rng)[0]
            if np.isfinite(T_meas):
                x_shadow[1] = T_meas

        # ---- experiment B: the twin observes and acts
        u = cooling_policy(x_twin[1])                           # <-- the automatic D->P arrow
        x_plant_b = rk4_step(pack_derivatives, t, x_plant_b, dt, I, u, T_amb, TRUE)
        x_twin = rk4_step(pack_derivatives, t, x_twin, dt, I, u, T_amb, BELIEF)
        if (k + 1) % int(sync_every / dt) == 0:
            T_meas = read_sensor(t, [x_plant_b[1]], TEMP_SENSOR, rng)[0]
            if np.isfinite(T_meas):
                x_twin[1] = T_meas
        u_log[k] = u

        for name, x in (("plant_a", x_plant_a), ("plant_b", x_plant_b), ("model", x_model),
                        ("shadow", x_shadow), ("twin", x_twin)):
            x[0] = np.clip(x[0], 0, 1)
            log[name][k + 1] = x
    u_log[-1] = u_log[-2]
    out = {k: pd.DataFrame(v, columns=["SoC", "T", "Q", "R"]).assign(t=ts) for k, v in log.items()}
    out["u"] = pd.Series(u_log, index=ts)
    return out

L = run_ladder()
print("simulation complete:", len(L["plant_a"]), "steps")

**Python note.** `%` gives the remainder of a division: `7 % 3` is 1 and `240 % 120` is 0. `x_plant_a.copy()` makes an independent copy of an array. Without `.copy()`, `x_model = x_plant_a` would only give the same array a second name, and every step of the pack would also move the model. `np.random.default_rng(RANDOM_STATE)` creates a random number generator, and the fixed seed makes the sensor noise the same in every run. `{k: np.zeros((n + 1, 4)) for k in (...)}` is a dictionary comprehension: It creates a table of zeros for each of the five names in the round brackets.

In [ ]:
err_model = L["model"]["T"] - L["plant_a"]["T"]
err_shadow = L["shadow"]["T"] - L["plant_a"]["T"]
rmse = lambda e: float(np.sqrt(np.mean(e ** 2)))

fig, ax = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True)
ax[0].plot(L["plant_a"].t / 60, L["plant_a"]["T"], lw=2.2, color="#111111", label="physical pack")
ax[0].plot(L["model"].t / 60, L["model"]["T"], lw=1.4, ls="--", color="#c0392b",
           label=f"digital model  (no arrows)   RMSE {rmse(err_model):.2f} C")
ax[0].plot(L["shadow"].t / 60, L["shadow"]["T"], lw=1.4, color="#2980b9",
           label=f"digital shadow (one arrow)   RMSE {rmse(err_shadow):.2f} C")
ax[0].set_ylabel("cell temperature [C]"); ax[0].legend(fontsize=8, loc="lower right")
ax[0].set_title("Same plant, same physics, different data flow")

ax[1].plot(L["plant_a"].t / 60, L["plant_a"]["T"], lw=2.2, color="#111111",
           label="physical pack, uncontrolled")
ax[1].plot(L["plant_b"].t / 60, L["plant_b"]["T"], lw=2.2, color="#27ae60",
           label="physical pack, driven by the digital twin")
ax[1].plot(L["u"].index / 60, T_AMB + 12 * L["u"].values, lw=1.0, color="#f39c12",
           label="fan duty (scaled)")
ax[1].set_xlabel("time [min]"); ax[1].set_ylabel("cell temperature [C]")
ax[1].legend(fontsize=8, loc="lower right")
ax[1].set_title("The second arrow changes the asset")
plt.tight_layout(); plt.show()

**Python note.** `rmse = lambda e: ...` defines a short function in one line, without `def`: It takes an array of errors `e` and returns their root mean square. `sharex=True` gives the two stacked panels one common time axis.

**Observe and reflect.** In the upper panel the shadow jumps back to the pack at every reading and moves away from it between the readings. In which direction does it move, and does this direction agree with the sign of the drift rate $v$ in the lecture?

## 3. What each arrow buys

*Lecture: [What each arrow buys](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#what-each-arrow-buys)*

**From the formulas to the code.** In `report`, `e = virt["T"] - plant["T"]` is the error $e_k = \hat{T}_k - T_k$ for each of the 7201 stored seconds. `rmse_T` is its RMSE, and `max_err` is the largest error, $\max_k |e_k|$. The model and the shadow are compared with `plant_a`, and the twin with `plant_b`, the pack that it controls. `plant_fade_Ah` is the capacity lost in the trip, $\Delta Q = Q_0 - Q$ at its end, and `trips_to_80pct` is $n_{80} = 0.2\, Q_0 / \Delta Q$.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "rmse" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-01/NB01_concept_and_architecture.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def report(name, virt, plant, fade_ref=None):
    e = virt["T"] - plant["T"]
    fade = TRUE.Q0 - plant["Q"].iloc[-1]
    return dict(level=name, rmse_T=rmse(e), max_err=float(np.abs(e).max()),
                plant_Tmax=float(plant["T"].max()), plant_fade_Ah=fade)

tbl = pd.DataFrame([
    report("Digital Model", L["model"], L["plant_a"]),
    report("Digital Shadow", L["shadow"], L["plant_a"]),
    report("Digital Twin", L["twin"], L["plant_b"]),
])
tbl["trips_to_80pct"] = 0.20 * TRUE.Q0 / tbl.plant_fade_Ah
print(tbl.round(3).to_string(index=False))

m, s, t = tbl.rmse_T
print(f"\nestimation error   model {m:.2f} C  ->  shadow {s:.2f} C   "
      f"({(1 - s / m) * 100:.0f}% reduction from one arrow)")
print(f"peak plant temp    {tbl.plant_Tmax[0]:.1f} C  ->  {tbl.plant_Tmax[2]:.1f} C   "
      f"(the second arrow)")
print(f"service life       {tbl.trips_to_80pct[0]:.0f}  ->  {tbl.trips_to_80pct[2]:.0f} trips "
      f"to 80% capacity  (+{(tbl.trips_to_80pct[2] / tbl.trips_to_80pct[0] - 1) * 100:.0f}%)")

**Python note.** `dict(level=name, rmse_T=rmse(e), ...)` builds a dictionary from names and values, one row of the table, and `pd.DataFrame([...])` turns the list of three such dictionaries into a table with one column per key. `tbl["trips_to_80pct"] = ...` adds a column computed from another one, and `m, s, t = tbl.rmse_T` gives the three values of the column `rmse_T` one name each.

The next cell computes by how many degrees the twin lowers the peak temperature: the largest temperature of the pack without control, `L["plant_a"]["T"].max()`, minus that of the pack under the twin.

In [ ]:
peak_drop = L["plant_a"]["T"].max() - L["plant_b"]["T"].max()
print(f"the twin lowers the peak by {peak_drop:.1f} degrees")

**Observe and reflect.** The twin lowers the peak by about 17 degrees, although its belief is not more accurate than that of the shadow. Why does a belief of the same quality lead to such a different pack? How would the peak change with a fan rule that starts at 40 instead of 42 degrees?

## 4. How often should the twin listen?

*Lecture: [How often should the twin listen?](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#how-often-should-the-twin-listen)*

**From the formulas to the code.** The loop calls `run_ladder` for eight values of the synchronisation interval $\Delta t_{\mathrm{sync}}$, the argument `sync_every` in seconds. For each value it stores the RMSE of the shadow, `shadow_RMSE_C`, and the peak of the pack that the twin controls, `twin_plant_Tmax_C`. The random generator `rng` of section 2 is not started again between the calls, so every run draws other noise, and a second run of the cell gives slightly different numbers. `semilogx` draws the horizontal axis on a logarithmic scale, on which 10, 100 and 1000 seconds lie equally far apart.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "peak_drop" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-01/NB01_concept_and_architecture.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
rows = []
for sync in (10, 30, 60, 120, 300, 600, 1200, 3600):
    Lk = run_ladder(sync_every=float(sync))
    rows.append((sync, rmse(Lk["shadow"]["T"] - Lk["plant_a"]["T"]),
                 float(Lk["plant_b"]["T"].max())))
sweep = pd.DataFrame(rows, columns=["sync_interval_s", "shadow_RMSE_C", "twin_plant_Tmax_C"])

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].semilogx(sweep.sync_interval_s, sweep.shadow_RMSE_C, "o-", color="#2980b9")
ax[0].set_xlabel("sync interval [s]"); ax[0].set_ylabel("shadow temperature RMSE [C]")
ax[0].set_title("How often must the twin listen?")
ax[1].semilogx(sweep.sync_interval_s, sweep.twin_plant_Tmax_C, "o-", color="#27ae60")
ax[1].axhline(L["plant_a"]["T"].max(), ls="--", lw=.9, color="#c0392b", label="uncontrolled")
ax[1].set_xlabel("sync interval [s]"); ax[1].set_ylabel("peak plant temperature [C]")
ax[1].set_title("The effect on the asset"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print(sweep.round(3).to_string(index=False))

**Python note.** `rows.append((sync, ..., ...))` adds a tuple of three values to the list `rows`, and `pd.DataFrame(rows, columns=[...])` turns the list of tuples into a table with one row per tuple and the given column names.

In [ ]:
plt.figure(figsize=(6.5, 3.2))
plt.semilogx(sweep["sync_interval_s"], sweep["shadow_RMSE_C"], "o-", color="#2980b9")
plt.xlabel("synchronisation interval (s, log scale)"); plt.ylabel("error of the belief (C)")
plt.title("too rare: the model drifts; too often: the sensor noise is copied", fontsize=9); plt.show()

**Exercise A.** Which synchronisation interval gives the smallest error? Store it, in seconds, in `best_sync`.

In [ ]:
best_sync = None   # your answer

check("Exercise A", best_sync)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
best_sync = int(sweep.loc[sweep["shadow_RMSE_C"].idxmin(), "sync_interval_s"])

check("Exercise A", best_sync)

**Observe and reflect.** The error of the shadow is lowest at one of the short intervals, while the peak of the controlled pack hardly changes up to 20 minutes. Why does the asset care less about the interval than the belief does? Which column of the table would decide the interval for a twin whose only task is to keep the pack below 46 °C?

## 5. The reference architecture in code &#91;[3](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#ref-3)&#93;

*Lecture: [A reference architecture](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#a-reference-architecture)*

**Python move: classes, as in the Python step.** Here five objects of five classes work together in one loop.

**From the formulas to the code.** The five classes are the five parts of the architecture. `orchestrate` calls them once per second in the order of the lecture: `phys.step`, `conn.poll`, `virt.step`, `virt.correct`, `svc.evaluate`, `phys.actuate` and `store.put`. The line `self.x[1] += gain * (obs["T"] - self.x[1])` is the correction $\hat{T} \leftarrow \hat{T} + K (y_k - \hat{T})$, with $K = 1$ as the default value of `gain`. `poll` returns `None` until `period` seconds have passed since the last reading, so the first reading is taken at $t = 0$, and `syncs` counts the polls that returned a reading. The four `lambda` functions are the services of the lecture's table, and each receives the belief `virt.x`, never the true state `phys.x`. The virtual entity starts from the datasheet resistance `BELIEF.R0`. Of the fields `drift_per_hour` and `latency_s` of `SensorSpec`, `read_sensor` applies only the drift, so readings arrive without delay. The printed rows are 900 seconds apart, half a swing of the current, which is why the column `I` shows 40.0 in every row. The docstring of `PhysicalEntity` names OPC UA (Open Platform Communications Unified Architecture) and MQTT (Message Queuing Telemetry Transport), two protocols through which industrial machines exchange data.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "best_sync" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-01/NB01_concept_and_architecture.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
class PhysicalEntity:
    """Stands in for the asset. In deployment this class is replaced by an OPC-UA / MQTT client
    and nothing else in the architecture changes. This substitutability is the design goal."""
    def __init__(self, params, x0, T_amb=25.0, cycle="urban"):
        self.p, self.x, self.T_amb, self.cycle, self.t = params, np.array(x0, float), T_amb, cycle, 0.0
        self.u = 0.0
    def step(self, dt):
        I = float(duty_cycle(self.t, self.cycle))
        self.x = rk4_step(pack_derivatives, self.t, self.x, dt, I, self.u, self.T_amb, self.p)
        self.x[0] = np.clip(self.x[0], 0, 1); self.t += dt
        return I
    def actuate(self, u):        # the digital -> physical arrow
        self.u = float(np.clip(u, 0, 1))

class Connection:
    """Everything that degrades between the asset and the twin: noise, bias, loss, delay, rate."""
    def __init__(self, spec, period_s=120.0, seed=RANDOM_STATE):
        self.spec, self.period, self.rng, self.last = spec, period_s, np.random.default_rng(seed), -1e9
    def poll(self, t, truth):
        if t - self.last < self.period:
            return None
        self.last = t
        keys = list(truth)
        vals = read_sensor(t, [truth[k] for k in keys], self.spec, self.rng)
        return {k: v for k, v in zip(keys, vals) if np.isfinite(v)}

class DataStore:
    """Append-only history. The twin's memory, and the auditor's evidence."""
    def __init__(self):
        self.rows = []
    def put(self, row):
        self.rows.append(row)
    def frame(self):
        return pd.DataFrame(self.rows)

class VirtualEntity:
    """The model, its beliefs, and its correction rule."""
    def __init__(self, params, x0):
        self.p, self.x, self.t = params, np.array(x0, float), 0.0
    def step(self, dt, I, u, T_amb=25.0):
        self.x = rk4_step(pack_derivatives, self.t, self.x, dt, I, u, T_amb, self.p)
        self.x[0] = np.clip(self.x[0], 0, 1); self.t += dt
    def correct(self, obs, gain=1.0):
        if obs and "T" in obs:
            self.x[1] += gain * (obs["T"] - self.x[1])       # simplest possible observer

class Services:
    """The purpose of the twin. Every service that the business asked for is defined here and nowhere else."""
    def __init__(self, rules):
        self.rules = rules
    def evaluate(self, t, x):
        return {k: f(t, x) for k, f in self.rules.items()}

print("five parts of the architecture, five classes:")
print("PhysicalEntity, Connection, VirtualEntity, DataStore, Services")

**Python note.** As in the Python step, `__init__` stores the values of a new object on `self`, and the other methods read and change them, as `step` does with `self.x` and `self.t`. `self.t += dt` is short for `self.t = self.t + dt`. In `poll`, `list(truth)` gives the keys of the dictionary `truth`, and `np.isfinite(v)` is false for a reading lost on the way, which `read_sensor` marks as `nan`, not a number, so the returned dictionary holds only the readings that arrived. `evaluate` calls every function in `self.rules` with the time and the belief and returns the results under the same keys. `DataStore` keeps the history as a list of dictionaries: The next cell builds the dictionary `row` of every step, adds the outputs of the services with `row.update(out)` and stores it with `store.put(row)`.

In [ ]:
def orchestrate(T_end=7200.0, dt=1.0, gain=1.0, authority=True, sensor=TEMP_SENSOR,
                period=120.0, seed=RANDOM_STATE):
    """One loop. This shape does not change when the plant becomes real."""
    phys = PhysicalEntity(TRUE, [0.95, T_AMB + 1, TRUE.Q0, TRUE.R0], T_amb=T_AMB, cycle=CYCLE)
    virt = VirtualEntity(BELIEF, [0.95, T_AMB + 1, BELIEF.Q0, BELIEF.R0])
    conn = Connection(sensor, period_s=period, seed=seed)
    store = DataStore()
    svc = Services({
        "fan_duty":    lambda t, x: float(np.clip((x[1] - 42.0) / 6.0, 0, 1)),
        "overheat":    lambda t, x: bool(x[1] > 46.0),
        "range_min":   lambda t, x: float(x[0] * x[2] / max(duty_cycle(t, CYCLE), 1e-6) * 60),
        "health_pct":  lambda t, x: float(100 * x[2] / TRUE.Q0),
    })
    for k in range(int(T_end / dt)):
        t = k * dt
        I = phys.step(dt)                                    # the world moves
        obs = conn.poll(t, {"T": phys.x[1], "SoC": phys.x[0]})  # physical -> digital
        virt.step(dt, I, phys.u, T_amb=T_AMB)                # the belief moves
        virt.correct(obs, gain)                              # the belief is corrected
        out = svc.evaluate(t, virt.x)                        # services read the belief
        if authority:
            phys.actuate(out["fan_duty"])                    # digital -> physical
        row = {"t": t, "I": I, "u": phys.u, "T_true": phys.x[1], "T_belief": virt.x[1],
               "SoC_true": phys.x[0], "SoC_belief": virt.x[0], "synced": obs is not None}
        row.update(out)                                      # the outputs of the services
        store.put(row)                                       # one row of the history
    return store.frame()

run = orchestrate()
print(run[["t", "I", "u", "T_true", "T_belief", "overheat", "health_pct"]]
      .iloc[::900].round(3).to_string(index=False))
print(f"\nbelief error  RMSE {rmse(run.T_belief - run.T_true):.3f} C   "
      f"max {np.abs(run.T_belief - run.T_true).max():.3f} C")
print(f"syncs: {int(run.synced.sum())} in {int(run.t.iloc[-1])} s")

**Observe and reflect.** Every reading moves the belief by the full residual, because `gain=1.0`. Run `orchestrate(gain=0.5)` in a new cell and compare the RMSE of its belief with the value above. Which error of the sensor does a gain below 1 reduce, and which can no gain remove?

## 6. Grading the twin against scenarios &#91;[5](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#ref-5)&#93;

*Lecture: [Testing before hardware exists](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#testing-before-hardware-exists)*

**From the formulas to the code.** Each entry of `scenarios` is a set of keyword arguments for `orchestrate`: another `SensorSpec`, another `period` or `authority=False`. `truly_hot_s` counts the rows in which the true temperature `T_true` is above 46, and `twin_says_hot_s` the rows in which the service `overheat` was true, that is, in which the belief was above 46. With one row per second both counts are seconds, the sums $t_{\mathrm{true}}$ and $t_{\mathrm{rep}}$ of the lecture. `belief_RMSE_C` is the RMSE of `T_belief - T_true`, `plant_Tmax_C` is the peak of the true temperature, and `mean_fan` is the mean of the fan duty $u$. In the bias scenario, `bias=-5.0` is the $b$ of the sensor formula, and in the packet-loss scenario `dropout=0.70` is the probability that a reading is lost.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "run" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-01/NB01_concept_and_architecture.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
# A known ground truth allows the twin to be graded before hardware exists.
scenarios = {
    "nominal":           dict(),
    "sensor bias -5 C":  dict(sensor=SensorSpec(noise_sd=0.35, bias=-5.0, quantum=0.25)),
    "70% packet loss":   dict(sensor=SensorSpec(noise_sd=0.35, bias=0.2, dropout=0.70)),
    "slow link (20 min)": dict(period=1200.0),
    "no authority":      dict(authority=False),
}
rows = []
for name, kw in scenarios.items():
    r = orchestrate(**kw)
    rows.append(dict(scenario=name,
                     belief_RMSE_C=rmse(r.T_belief - r.T_true),
                     plant_Tmax_C=float(r.T_true.max()),
                     twin_says_hot_s=int(r.overheat.sum()),
                     truly_hot_s=int((r.T_true.values > 46.0).sum()),
                     mean_fan=float(r.u.mean())))
grade = pd.DataFrame(rows)
print(grade.round(3).to_string(index=False))
print("\n'twin_says_hot_s' = seconds for which the belief of the twin was over 46 C.")
print("'truly_hot_s'     = seconds for which the pack was over 46 C.")
b = grade.set_index("scenario")
print(f"\nbias scenario: pack over 46 C for {b.loc['sensor bias -5 C','truly_hot_s']:.0f} s, "
      f"twin reported it for {b.loc['sensor bias -5 C','twin_says_hot_s']:.0f} s.")

**Python note.** `orchestrate(**kw)` passes the entries of the dictionary `kw` as keyword arguments, that is, as arguments given with their names: The slow link, `dict(period=1200.0)`, runs `orchestrate(period=1200.0)`, and the empty `dict()` of the nominal scenario keeps every default value of `orchestrate`.

In [ ]:
ax = grade.set_index("scenario")[["truly_hot_s", "twin_says_hot_s"]].plot.barh(figsize=(7.5, 3.6), color=["#c0392b", "#2980b9"])
ax.set_xlabel("seconds above 46 C"); ax.set_title("true seconds above the limit and seconds reported by the twin", fontsize=9)
plt.tight_layout(); plt.show()

**Exercise B.** How many seconds is the pack truly above 46 C under the scenario `sensor bias -5 C`? Store the number in `hidden_s`.

In [ ]:
hidden_s = None   # your answer

check("Exercise B", hidden_s)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
hidden_s = float(grade.set_index("scenario").loc["sensor bias -5 C", "truly_hot_s"])

check("Exercise B", hidden_s)

**Observe and reflect.** In the scenario `slow link (20 min)` the twin reports fewer hot seconds than the pack truly spends above the limit, and in `no authority` the two counts are almost equal. Explain both from the moment at which the belief learns about the temperature of the pack.

## 7. Your asset: the application switch

*Lecture: [Your asset](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#your-asset)*

Change `ASSET` to `server_rack`, `motor` or `greenhouse` and run the cell again.

**From the formulas to the code.** `run_asset` is the explicit Euler step for the general model of the lecture, `T[k + 1] = T[k] + dt * (...) / p["C"]`, with `a["power"](t[k])` as the heat input $P(t)$. `run_asset(a)` is the asset without cooling, and `run_asset(a, params=belief)` is the digital model, whose heat capacity is 15 percent and whose passive conductance is 30 percent above the true values. In `shadow`, `Tp` is the true temperature and `Tb` the belief, and every `sync` seconds the line `Tb[k + 1] = Tp[k + 1] + sd * rng.normal()` overwrites the belief with a reading whose noise has the standard deviation `sd`. The `lambda` passed as `controller` is the fan rule $u = \mathrm{clip}((\hat{T} - (T_{\mathrm{lim}} - 4)) / 4, 0, 1)$. `rmse` is the RMSE, and `over` multiplies the number of steps above the limit by the step length to obtain seconds.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "hidden_s" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-01/NB01_concept_and_architecture.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
ASSET = "battery"        # choose: battery, server_rack, motor or greenhouse

ASSETS = {
    "battery":     dict(name="battery pack", C=9000.0, hA0=2.2, hA1=12.0, T_amb=35.0, T0=36.0, limit=46.0,
                        horizon=7200.0, dt=5.0, power=lambda t: (40 + 6 * np.sin(2 * np.pi * t / 1800)) ** 2 * 0.045),
    "server_rack": dict(name="server rack", C=1.2e5, hA0=40.0, hA1=400.0, T_amb=24.0, T0=26.0, limit=35.0,
                        horizon=7200.0, dt=10.0, power=lambda t: 3000 + 1500 * np.maximum(0, np.sin(2 * np.pi * t / 3600))),
    "motor":       dict(name="motor winding", C=4000.0, hA0=1.5, hA1=8.0, T_amb=30.0, T0=32.0, limit=90.0,
                        horizon=5400.0, dt=5.0, power=lambda t: (20 + 8 * np.sin(2 * np.pi * t / 1200)) ** 2 * 0.3),
    "greenhouse":  dict(name="greenhouse air", C=3.0e6, hA0=500.0, hA1=4000.0, T_amb=18.0, T0=20.0, limit=32.0,
                        horizon=86400.0, dt=60.0, power=lambda t: 30000 * np.maximum(0, np.sin(2 * np.pi * t / 86400))),
}

def run_asset(a, controller=None, params=None, sensor_sd=0.0, seed=0, T_end=None, dt=None):
    """Explicit Euler for dT/dt = (P(t) - (hA0 + hA1 u)(T - T_amb)) / C; the controller maps a reading to a duty u."""
    p = dict(a); p.update(params or {}); dt = dt or a["dt"]; n = int((T_end or a["horizon"]) / dt)
    t = np.arange(n) * dt; T = np.empty(n); u = np.zeros(n); rng = np.random.default_rng(seed); T[0] = a["T0"]
    for k in range(n - 1):
        reading = T[k] + sensor_sd * rng.normal()
        u[k] = controller(reading) if controller else 0.0
        T[k + 1] = T[k] + dt * (a["power"](t[k]) - (p["hA0"] + p["hA1"] * u[k]) * (T[k] - p["T_amb"])) / p["C"]
    return t, T, u


a = ASSETS[ASSET]; rng = np.random.default_rng(RANDOM_STATE)
belief = dict(C=a["C"] * 1.15, hA0=a["hA0"] * 1.3)                       # datasheet values, which differ from the true ones
sync = max(a["dt"], round(a["horizon"] / 60 / a["dt"]) * a["dt"])         # sixty updates over the horizon
t, T_plant, _ = run_asset(a)                                              # the asset without cooling
_, T_model, _ = run_asset(a, params=belief)                               # digital model: no data
def shadow(controller=None, sd=0.35 * (a["limit"] - a["T_amb"]) / 11):
    """A belief that is overwritten by a noisy reading at every synchronisation, optionally acting on the asset."""
    n = len(t); Tp = np.empty(n); Tb = np.empty(n); u = np.zeros(n); Tp[0] = Tb[0] = a["T0"]
    for k in range(n - 1):
        u[k] = controller(Tb[k]) if controller else 0.0
        heat = a["power"](t[k])
        Tp[k + 1] = Tp[k] + a["dt"] * (heat - (a["hA0"] + a["hA1"] * u[k]) * (Tp[k] - a["T_amb"])) / a["C"]
        Tb[k + 1] = Tb[k] + a["dt"] * (heat - (belief["hA0"] + a["hA1"] * u[k]) * (Tb[k] - a["T_amb"])) / belief["C"]
        if (k + 1) % int(sync / a["dt"]) == 0:
            Tb[k + 1] = Tp[k + 1] + sd * rng.normal()
    return Tp, Tb, u
Tp_s, Tb_s, _ = shadow()
Tp_t, Tb_t, u_t = shadow(controller=lambda Tb: float(np.clip((Tb - (a["limit"] - 4)) / 4, 0, 1)))
rmse = lambda e: float(np.sqrt(np.mean(np.asarray(e) ** 2)))
over = lambda T_: float((T_ > a["limit"]).sum() * a["dt"])
print(f"{a['name']}: belief error, digital model {rmse(T_model - T_plant):.2f}, digital shadow {rmse(Tb_s - Tp_s):.2f}")
print(f"peak without control {T_plant.max():.1f}, with the twin {Tp_t.max():.1f}; time above the limit "
      f"{over(T_plant) / 60:.0f} min without control, {over(Tp_t) / 60:.0f} min with the twin")
scale = 3600 if a["horizon"] > 20000 else 60; unit = "h" if scale == 3600 else "min"
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(t / scale, T_plant, color="k", label="asset"); ax[0].plot(t / scale, T_model, "--", label="digital model")
ax[0].plot(t / scale, Tb_s, label="digital shadow", alpha=.8); ax[0].legend(fontsize=8); ax[0].set_title("what each counterpart believes", fontsize=9)
ax[1].plot(t / scale, T_plant, color="k", label="no control"); ax[1].plot(t / scale, Tp_t, color="#27ae60", label="with the twin")
ax[1].axhline(a["limit"], ls="--", color="#c0392b", lw=1); ax[1].legend(fontsize=8); ax[1].set_title("what the twin changes", fontsize=9)
for x in ax: x.set_xlabel(f"time ({unit})")
plt.tight_layout(); plt.show()

**Python note.** `ASSETS` is a dictionary of dictionaries: `ASSETS["motor"]["C"]` first selects the motor and then its heat capacity. The entry `power=lambda t: ...` stores a function as a value, so that each asset carries its own heat input, and `a["power"](t[k])` calls it at the time `t[k]`.

The next cell computes how many minutes above its limit the twin saves for the chosen asset. `over` is the function of the cell above, which multiplies the number of time steps above the limit by the length of a step.

In [ ]:
saved_min = (over(T_plant) - over(Tp_t)) / 60
print(f"{a['name']}: the twin saves {saved_min:.0f} minutes above the limit")

**Observe and reflect.** Switch between the four assets and compare the saved minutes. Which asset can the twin protect completely, and which property of the server rack, visible in the table of the lecture, limits what the twin can do there?

## 8. Going further (optional): a biased sensor in control &#91;[7](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#ref-7)&#93;

*Lecture: [Going further (optional)](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lecture.html#going-further-optional)*

**From the formulas to the code.** The loop calls `orchestrate` for six values of the sensor bias $b$, each time with a noise of 0.35 degrees, a rounding step of 0.25 degrees and no loss. `truly_hot_s` and `reported_hot_s` count the rows in which `T_true` and `T_belief` are above 46, the sums $t_{\mathrm{true}}$ and $t_{\mathrm{rep}}$ of the lecture in seconds, and the plot divides them by 60 to show minutes. `peak_C` is the highest true temperature of the run. In `explore`, the sliders `sync_s`, `bias_C`, `noise_C` and `loss` set the synchronisation interval, the bias, the standard deviation of the noise and the probability of packet loss for one run of the same loop.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "saved_min" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/digital-twin-veltech-NB-lecture/main/days/day-01/NB01_concept_and_architecture.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-8":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
# A biased sensor drives the twin. For each bias, the loop counts the seconds above the limit and the seconds that the twin reports.
rows = []
for bias in (-6, -4, -2, 0, 2, 4):
    log_b = orchestrate(sensor=SensorSpec(noise_sd=0.35, bias=float(bias), quantum=0.25), period=120.0)
    rows.append(dict(bias_C=bias, truly_hot_s=float((log_b["T_true"] > 46).sum()), reported_hot_s=float((log_b["T_belief"] > 46).sum()),
                     peak_C=float(log_b["T_true"].max())))
bias_tab = pd.DataFrame(rows); print(bias_tab.to_string(index=False))
plt.figure(figsize=(6.5, 3.2)); plt.plot(bias_tab.bias_C, bias_tab.truly_hot_s / 60, "o-", color="#c0392b", label="truly above 46 C")
plt.plot(bias_tab.bias_C, bias_tab.reported_hot_s / 60, "s-", color="#2980b9", label="reported above 46 C")
plt.xlabel("sensor bias (C)"); plt.ylabel("minutes"); plt.legend(fontsize=8); plt.show()

**Observe and reflect.** A bias of +2 degrees makes the twin report overheating that never happened, and a bias of $-2$ degrees hides real overheating. Which of the two errors costs the owner of the pack more, and which is more dangerous for the people near the pack?

**Explore.** Move the controls and watch the result update. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W
def explore(sync_s=120, bias_C=0.0, noise_C=0.35, loss=0.0):
    spec = SensorSpec(noise_sd=noise_C, bias=bias_C, quantum=0.25, dropout=loss)
    log = orchestrate(sensor=spec, period=float(sync_s))
    over = float((log["T_true"] > 46).sum()); rep = float((log["T_belief"] > 46).sum())
    plt.figure(figsize=(8, 3)); plt.plot(log["t"] / 60, log["T_true"], label="true"); plt.plot(log["t"] / 60, log["T_belief"], label="belief")
    plt.axhline(46, color="red", ls="--", lw=1); plt.xlabel("time (min)"); plt.ylabel("°C"); plt.legend(); plt.show()
    print(f"truly over 46 °C: {over:.0f} s, reported over 46 °C: {rep:.0f} s")
W.interact(explore, sync_s=W.IntSlider(value=120, min=10, max=1200, step=10),
           bias_C=W.FloatSlider(value=0.0, min=-6.0, max=6.0, step=0.5),
           noise_C=W.FloatSlider(value=0.35, min=0.0, max=1.5, step=0.05), loss=W.FloatSlider(value=0.0, min=0.0, max=0.9, step=0.05));

**Python note.** `W.interact(explore, sync_s=W.IntSlider(...), ...)` draws one slider for each argument of `explore` and runs the function again with the new values whenever a slider moves.

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Replace the in-memory data store with an append-only CSV file and show that the orchestration loop does not need to change. |
| Electronics and Communication Engineering | Add a transmission delay of 30 seconds to the connection and measure its effect on the peak temperature of the twin-controlled pack. |
| Electrical and Electronics Engineering | Change the duty cycle to an urban profile with short bursts of high current and repeat the maturity ladder. Which arrow matters more now? |
| Mechanical Engineering | Double the thermal mass of the pack and explain, with the energy balance of the lecture, why the optimal synchronisation interval moves. |
| Biomedical Engineering | Sketch the five components of the reference architecture for a twin of a patient on an infusion pump, and name the most dangerous data flow. |
| Civil Engineering | Classify three monitoring systems of bridges or buildings that you can find in the literature as models, shadows or twins, by their data flows. |
| Aeronautical Engineering | Design a scenario for the scenario table in which the twin should refuse to act, and state the condition it should check. |
| Biotechnology | Write the state, the inputs and the time scales of a twin of a bioreactor in the style of the pack description of the lecture. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/digital-twin-veltech-NB-lecture/days/day-01/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/digital-twin-veltech-NB-lecture/tree/main/days/day-01).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>